# PROJECT-2 Implied Volatility Surface Prediction 
### Finance Club, IIT Roorkee: Open Projects 2026
# EKANSH SINGH 23112037


### 1 The Model: Separate Linear Interpolation + Dual Convexity Correction + Time-Decay Blending
- Separate Interpolation:  IN this project I majorily do Linear interpolation of PE (strikes 23800–25100) and CE (strikes 25200–26500) separately.
- Dual Convexity Correction: I  subtract a small correction for interpolated points (tuned to $\alpha_{\text{pe}} = -1.0 \times 10^{-8}$ and $\alpha_{\text{ce}} = -6.0 \times 10^{-8}$), and add an upward correction for extrapolated points (tuned to $\beta = 4.5 \times 10^{-8}$).
- Time-Decay Blending: We dynamically blend with the last observed past value using exponential decay parameters (base weight $0.015$, decay $\gamma = 0.70$).

This is giving  local validation MSE.

**In this cell finding dataset from kaggle competiotion link of finanace club**

In [5]:
import os
import re
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d


SEPARATOR = "||"
OUTPUT_FILLED_PATH = "filled_dataset.csv"
OUTPUT_SUBMISSION_PATH = "submission.csv"

dataset_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file in ['dataset.csv', 'dataset (2).csv']:
            dataset_path = os.path.join(root, file)
            break

if dataset_path is None:
    
    dataset_path = 'dataset.csv' if os.path.exists('dataset.csv') else 'dataset (2).csv'

print(f"Found dataset at: {dataset_path}")

Found dataset at: /kaggle/input/competitions/finclub-open-project-26/dataset.csv


## 2. Data Loading & Preprocessing
### Data Preparation

I begin by loading the options dataset and identifying the Call (CE) and Put (PE) contracts. The strike price is extracted from each contract name so that the contracts can be sorted properly. Having the contracts in the correct strike order is important for accurately estimating missing implied volatility values.

In [6]:

df = pd.read_csv(dataset_path)
print(f"Dataset Shape: {df.shape}")
print(f"Total cells: {df.shape[0] * (df.shape[1] - 2)}")
print(f"Missing cells (target count): {df.isnull().sum().sum()}")


option_cols = [c for c in df.columns if c not in ["datetime", "underlying_price"]]
ce_cols = [c for c in option_cols if c.endswith("CE")]
pe_cols = [c for c in option_cols if c.endswith("PE")]


def get_strike(col):
    m = re.match(r"NIFTY(\d+[A-Z]+\d{2})(\d+)(CE|PE)", col)
    if m:
        return int(m.group(2))
    raise ValueError(f"Could not parse column name: {col}")


pe_strikes = np.array([get_strike(c) for c in pe_cols])
sorted_pe_idx = np.argsort(pe_strikes)
pe_cols_sorted = [pe_cols[i] for i in sorted_pe_idx]
pe_strikes_sorted = pe_strikes[sorted_pe_idx]

ce_strikes = np.array([get_strike(c) for c in ce_cols])
sorted_ce_idx = np.argsort(ce_strikes)
ce_cols_sorted = [ce_cols[i] for i in sorted_ce_idx]
ce_strikes_sorted = ce_strikes[sorted_ce_idx]

print(f"Sorted PE strikes: {pe_strikes_sorted}")
print(f"Sorted CE strikes: {ce_strikes_sorted}")

Dataset Shape: (975, 30)
Total cells: 27300
Missing cells (target count): 5460
Sorted PE strikes: [23800 23900 24000 24100 24200 24300 24400 24500 24600 24700 24800 24900
 25000 25100]
Sorted CE strikes: [25200 25300 25400 25500 25600 25700 25800 25900 26000 26100 26200 26300
 26400 26500]


## 3. Surface Fitting (Linear Interpolation + Convexity Correction + Time-Decay Blending)
### Implied Volatility Prediction

To estimate the missing implied volatility values, I first treat CE and PE contracts separately and fit a volatility curve for each group. The curve is then used to fill missing values within the observed strike range and to estimate values beyond it when required. Historical volatility information is also incorporated, with recent observations receiving higher importance, helping improve the stability of the final predictions.

In [7]:

alpha_pe = -1.0e-8
alpha_ce = -6.0e-8
beta_left = 4.5e-8
beta_right = 4.5e-8
base_w_past = 0.015
gamma = 0.7

option_cols_sorted = pe_cols_sorted + ce_cols_sorted
X = df[option_cols_sorted].values


last_obs_val = np.full_like(X, np.nan)
last_obs_dist = np.full_like(X, np.nan)

for c_idx in range(X.shape[1]):
    last_val = np.nan
    last_r = np.nan
    for r_idx in range(X.shape[0]):
        last_obs_val[r_idx, c_idx] = last_val
        if not np.isnan(last_r):
            last_obs_dist[r_idx, c_idx] = r_idx - last_r
        if not np.isnan(X[r_idx, c_idx]):
            last_val = X[r_idx, c_idx]
            last_r = r_idx

filled_df = df.copy()


X_pe = df[pe_cols_sorted].values
X_pe_filled = X_pe.copy()
n_pe = len(pe_cols_sorted)

for i in range(len(df)):
    row = X_pe[i]
    missing = np.isnan(row)
    if not missing.any():
        continue
    valid = ~missing
    n_val = valid.sum()
    if n_val >= 2:
        f = interp1d(pe_strikes_sorted[valid], row[valid], kind='linear', fill_value='extrapolate')
        linear_pred = f(pe_strikes_sorted)
        
        first_idx = np.where(valid)[0][0]
        last_idx = np.where(valid)[0][-1]
        
        for col_idx in np.where(missing)[0]:
            past_val = last_obs_val[i, col_idx]
            d = last_obs_dist[i, col_idx]
            
            if first_idx < col_idx < last_idx:
                left_obs_idx = np.where(valid & (np.arange(n_pe) < col_idx))[0][-1]
                right_obs_idx = np.where(valid & (np.arange(n_pe) > col_idx))[0][0]
                left_dist = pe_strikes_sorted[col_idx] - pe_strikes_sorted[left_obs_idx]
                right_dist = pe_strikes_sorted[right_obs_idx] - pe_strikes_sorted[col_idx]
                
                correction = alpha_pe * left_dist * right_dist
                pred_val = linear_pred[col_idx] + correction
            else:
                if col_idx < first_idx:
                    dist = pe_strikes_sorted[first_idx] - pe_strikes_sorted[col_idx]
                    pred_val = linear_pred[col_idx] + beta_left * (dist ** 2)
                else:
                    dist = pe_strikes_sorted[col_idx] - pe_strikes_sorted[last_idx]
                    pred_val = linear_pred[col_idx] + beta_right * (dist ** 2)
                
            if not np.isnan(past_val) and not np.isnan(d):
                w_past = base_w_past * (gamma ** (d - 1))
                pred_val = (1.0 - w_past) * pred_val + w_past * past_val
            X_pe_filled[i, col_idx] = pred_val
    elif n_val == 1:
        X_pe_filled[i, missing] = row[valid][0]
    else:
        X_pe_filled[i, missing] = 0.15

filled_df[pe_cols_sorted] = X_pe_filled

X_ce = df[ce_cols_sorted].values
X_ce_filled = X_ce.copy()
n_ce = len(ce_cols_sorted)

for i in range(len(df)):
    row = X_ce[i]
    missing = np.isnan(row)
    if not missing.any():
        continue
    valid = ~missing
    n_val = valid.sum()
    if n_val >= 2:
        f = interp1d(ce_strikes_sorted[valid], row[valid], kind='linear', fill_value='extrapolate')
        linear_pred = f(ce_strikes_sorted)
        
        first_idx = np.where(valid)[0][0]
        last_idx = np.where(valid)[0][-1]
        
        for col_idx in np.where(missing)[0]:
            global_col_idx = n_pe + col_idx
            past_val = last_obs_val[i, global_col_idx]
            d = last_obs_dist[i, global_col_idx]
            
            if first_idx < col_idx < last_idx:
                left_obs_idx = np.where(valid & (np.arange(n_ce) < col_idx))[0][-1]
                right_obs_idx = np.where(valid & (np.arange(n_ce) > col_idx))[0][0]
                left_dist = ce_strikes_sorted[col_idx] - ce_strikes_sorted[left_obs_idx]
                right_dist = ce_strikes_sorted[right_obs_idx] - ce_strikes_sorted[col_idx]
                
                correction = alpha_ce * left_dist * right_dist
                pred_val = linear_pred[col_idx] + correction
            else:
                if col_idx < first_idx:
                    dist = ce_strikes_sorted[first_idx] - ce_strikes_sorted[col_idx]
                    pred_val = linear_pred[col_idx] + beta_left * (dist ** 2)
                else:
                    dist = ce_strikes_sorted[col_idx] - ce_strikes_sorted[last_idx]
                    pred_val = linear_pred[col_idx] + beta_right * (dist ** 2)
                
            
            if not np.isnan(past_val) and not np.isnan(d):
                w_past = base_w_past * (gamma ** (d - 1))
                pred_val = (1.0 - w_past) * pred_val + w_past * past_val
            X_ce_filled[i, col_idx] = pred_val
    elif n_val == 1:
        X_ce_filled[i, missing] = row[valid][0]
    else:
        X_ce_filled[i, missing] = 0.15

filled_df[ce_cols_sorted] = X_ce_filled
print("Finished filling the volatility surface.")

Finished filling the volatility surface.


## 4. Verification and Submission File Generation
### Submission Generation

As a final step, I verify that every missing value has been filled and that the dataset is ready for submission. The completed predictions are then organized according to the competition guidelines and exported as the final CSV file.

In [8]:

nan_count = filled_df.isnull().sum().sum()
print(f"Remaining NaNs in filled dataset: {nan_count}")
assert nan_count == 0, "There are still missing values in the dataset!"


filled_df.to_csv(OUTPUT_FILLED_PATH, index=False)
print(f"Saved filled dataset to: {OUTPUT_FILLED_PATH}")


feature_cols = [c for c in df.columns if c != "datetime"]
rows = []
for col in feature_cols:
    was_missing = df[col].isna()
    for idx in df.index[was_missing]:
        dt = df.loc[idx, "datetime"]
        uid = f"{dt}{SEPARATOR}{col}"
        val = filled_df.loc[idx, col]
        rows.append({"id": uid, "value": val})

solution = pd.DataFrame(rows, columns=["id", "value"])
solution = solution.sort_values("id").reset_index(drop=True)
solution.to_csv(OUTPUT_SUBMISSION_PATH, index=False)
print(f"Submission file successfully saved to: {OUTPUT_SUBMISSION_PATH} ({len(solution)} rows)")


print("\nFirst 5 rows of submission:")
print(solution.head())

Remaining NaNs in filled dataset: 0
Saved filled dataset to: filled_dataset.csv
Submission file successfully saved to: submission.csv (5460 rows)

First 5 rows of submission:
                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.163340
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.113130
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.100900
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.169945
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.159639
